# Probar un endpoint de Databricks Model Serving

Este notebook realiza una invocación REST usando `dataframe_split` y muestra las predicciones devueltas. El token se obtiene desde Databricks Secrets y nunca se imprime. El endpoint está configurado con scale-to-zero, por lo que la primera llamada puede tardar entre 15 y 30 segundos.

## Requisitos

- Endpoint activo de Databricks Model Serving.
- Secret scope con un token válido.
- Permiso para invocar el endpoint.
- Modelo compatible con las columnas Iris de este ejemplo.

`requests` y `pandas` suelen estar disponibles en Databricks Runtime. Si no lo están, instala `requests` y `pandas` como librerías del cluster o ejecuta `%pip install requests pandas` en una celda separada y reinicia Python.

In [ ]:
# Configuración mediante widgets de Databricks.
# El token se lee desde Secrets y no se guarda en el notebook.
dbutils.widgets.text("DATABRICKS_HOST", "https://dbc-f2dbc696-258a.cloud.databricks.com", "Workspace Databricks")
dbutils.widgets.text("DATABRICKS_ENDPOINT_NAME", "iris-random-forest", "Nombre del endpoint")
dbutils.widgets.text("DATABRICKS_SECRET_SCOPE", "mi-scope", "Secret scope")
dbutils.widgets.text("DATABRICKS_SECRET_KEY", "databricks-token", "Secret key del token")

HOST = dbutils.widgets.get("DATABRICKS_HOST").strip()
ENDPOINT_NAME = dbutils.widgets.get("DATABRICKS_ENDPOINT_NAME").strip()
SECRET_SCOPE = dbutils.widgets.get("DATABRICKS_SECRET_SCOPE").strip()
SECRET_KEY = dbutils.widgets.get("DATABRICKS_SECRET_KEY").strip()

if not HOST or "<workspace>" in HOST:
    raise ValueError("Configura DATABRICKS_HOST con el workspace real.")
if not ENDPOINT_NAME:
    raise ValueError("Configura DATABRICKS_ENDPOINT_NAME.")
if not SECRET_SCOPE or not SECRET_KEY:
    raise ValueError("Configura el secret scope y la secret key del token.")

In [ ]:
import json
from typing import Any

import pandas as pd
import requests

TIMEOUT_SEGUNDOS = 60
TOKEN = dbutils.secrets.get(scope=SECRET_SCOPE, key=SECRET_KEY)
if not TOKEN:
    raise ValueError("El secret configurado no contiene un token válido.")

host_limpio = HOST.removeprefix("https://").removeprefix("http://").rstrip("/")
endpoint_limpio = ENDPOINT_NAME.strip("/")
URL = f"https://{host_limpio}/serving-endpoints/{endpoint_limpio}/invocations"
HEADERS = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json",
}

print(f"Endpoint: {URL}")
print(f"Timeout: {TIMEOUT_SEGUNDOS} segundos")
print("Token cargado desde Databricks Secrets: sí (valor oculto)")

In [ ]:
columnas = [
    "SepalLengthCm",
    "SepalWidthCm",
    "PetalLengthCm",
    "PetalWidthCm",
]
datos = [
    [5.1, 3.5, 1.4, 0.2],
    [6.4, 3.2, 4.5, 1.5],
    [6.7, 3.1, 5.6, 2.4],
]
payload = {"dataframe_split": {"columns": columnas, "data": datos}}
print(json.dumps(payload, indent=2))
dataframe_entrada = pd.DataFrame(datos, columns=columnas)
display(dataframe_entrada)

## Invocación REST

La respuesta exitosa debe contener una clave `predictions`. Los errores muestran el código HTTP y el texto devuelto por Databricks.

In [ ]:
try:
    respuesta = requests.post(URL, headers=HEADERS, json=payload, timeout=TIMEOUT_SEGUNDOS)
except requests.Timeout as error:
    raise RuntimeError(f"La solicitud superó el timeout de {TIMEOUT_SEGUNDOS} segundos.") from error
except requests.ConnectionError as error:
    raise RuntimeError("No se pudo conectar con Databricks; revisa host y red.") from error

print(f"Código HTTP: {respuesta.status_code}")
if not respuesta.ok:
    raise RuntimeError(f"Databricks devolvió HTTP {respuesta.status_code}: {respuesta.text}")

try:
    respuesta_json: Any = respuesta.json()
except ValueError as error:
    raise RuntimeError(f"La respuesta no es JSON válido: {respuesta.text}") from error

if not isinstance(respuesta_json, dict) or "predictions" not in respuesta_json:
    raise RuntimeError("La respuesta no contiene la clave 'predictions'.")

predicciones = respuesta_json["predictions"]
if not isinstance(predicciones, list):
    raise RuntimeError("La clave 'predictions' no contiene una lista.")
if len(predicciones) != len(datos):
    raise RuntimeError(f"Se recibieron {len(predicciones)} predicciones para {len(datos)} filas.")

print(json.dumps(respuesta_json, indent=2, ensure_ascii=False))
print(f"Predicciones: {predicciones}")

In [ ]:
resultado = dataframe_entrada.copy()
resultado["prediccion"] = predicciones
display(resultado)

## Diagnóstico rápido

- **400**: revisa columnas, orden y tipos de entrada.
- **401**: el token es inválido, expiró o el secret es incorrecto.
- **403**: la identidad no tiene permiso para invocar el endpoint.
- **404**: revisa workspace y nombre exacto del endpoint.
- **429**: se alcanzó un límite de solicitudes; espera y reintenta.
- **500/503**: revisa el estado, logs y capacidad del endpoint.